# 04 — Advanced Analysis: Clustering & PCA

Tạo digital lifestyle archetypes, PCA user map, density contour và parallel coordinates.

In [8]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 120)

# Paths
PROJECT_ROOT = Path("..")
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "mobile_usage_behavioral_analysis.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
AGG_DIR = PROJECT_ROOT / "data" / "aggregated"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"

# Create necessary directories
for p in [PROCESSED_DIR, AGG_DIR, FIGURE_DIR]:
    p.mkdir(parents=True, exist_ok=True)
def title(t):
    print("\n" + "="*90); print(t); print("="*90)
def clean_cols(df):
    df = df.copy(); df.columns = [c.strip() for c in df.columns]; return df

# Load dataset and basic audit
title("Load processed dataset")
df = pd.read_csv(PROCESSED_DIR / "smartphone_features.csv")
print(df.shape)
display(df.head())



Load processed dataset
(1000, 21)


,User_ID,Age,Gender,Total_App_Usage_Hours,Daily_Screen_Time_Hours,Number_of_Apps_Used,Social_Media_Usage_Hours,Productivity_App_Usage_Hours,Gaming_App_Usage_Hours,Location,Age_Group,Screen_Time_Segment,Dominant_Lifestyle,Category_Usage_Sum,Entertainment_Hours,Screen_to_App_Gap,App_Fragmentation_Score,Category_vs_Total_Diff,Social_Ratio,Productivity_Ratio,Gaming_Ratio
0,1,56,Male,2.61,7.15,24,4.43,0.55,2.40,Los Angeles,55–59,Moderate (4–8h),Social Enthusiast,7.38,6.83,4.54,0.108750,4.77,0.600271,0.074526,0.325203
1,2,46,Male,2.13,13.79,18,4.67,4.42,2.43,Chicago,45–54,Extreme (>12h),Social Enthusiast,11.52,7.10,11.66,0.118333,9.39,0.405382,0.383681,0.210938
2,3,32,Female,7.28,4.50,11,4.58,1.71,2.83,Houston,25–34,Moderate (4–8h),Social Enthusiast,9.12,7.41,-2.78,0.661818,1.84,0.502193,0.187500,0.310307
3,4,25,Female,1.20,6.29,21,3.18,3.42,4.58,Phoenix,25–34,Moderate (4–8h),Mobile Gamer,11.18,7.76,5.09,0.057143,9.98,0.284436,0.305903,0.409660
4,5,38,Male,6.31,12.59,14,3.15,0.13,4.00,New York,35–44,Extreme (>12h),Mobile Gamer,7.28,7.15,6.28,0.450714,0.97,0.432692,0.017857,0.549451


In [9]:
# Prepare modeling matrix
title("Prepare modeling matrix")
features = ["Age", "Daily_Screen_Time_Hours", "Total_App_Usage_Hours", "Number_of_Apps_Used", "Social_Media_Usage_Hours", "Productivity_App_Usage_Hours", "Gaming_App_Usage_Hours", "Entertainment_Hours", "App_Fragmentation_Score"]
X = df[features].replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
display(X.describe().T.round(3))
print("X_scaled:", X_scaled.shape)



Prepare modeling matrix


,count,mean,std,min,25%,50%,75%,max
Age,1000.0,38.745,12.187,18.000,28.000,40.000,50.000,59.000
Daily_Screen_Time_Hours,1000.0,7.696,3.714,1.010,4.530,7.880,10.910,14.000
Total_App_Usage_Hours,1000.0,6.406,3.135,1.000,3.590,6.455,9.122,11.970
Number_of_Apps_Used,1000.0,16.647,7.620,3.000,10.000,17.000,23.000,29.000
Social_Media_Usage_Hours,1000.0,2.456,1.440,0.000,1.200,2.445,3.672,4.990
Productivity_App_Usage_Hours,1000.0,2.495,1.443,0.000,1.282,2.435,3.710,5.000
Gaming_App_Usage_Hours,1000.0,2.475,1.450,0.010,1.220,2.455,3.782,5.000
Entertainment_Hours,1000.0,4.932,2.050,0.080,3.470,4.970,6.370,9.760
App_Fragmentation_Score,1000.0,0.540,0.531,0.044,0.230,0.389,0.628,3.653


X_scaled: (1000, 9)


In [10]:
# Evaluate cluster count
title("Evaluate cluster count")
rows = []
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = km.fit_predict(X_scaled)
    rows.append({"k": k, "Inertia": km.inertia_, "Silhouette": silhouette_score(X_scaled, labels)})
eval_df = pd.DataFrame(rows)
display(eval_df.round(4))

# Visualize evaluation metrics
px.line(eval_df, x="k", y="Inertia", markers=True, title="Elbow method").show()
px.line(eval_df, x="k", y="Silhouette", markers=True, title="Silhouette score").show()
print("For storytelling, k=4 is used to create interpretable digital lifestyle archetypes.")



Evaluate cluster count


,k,Inertia,Silhouette
0,2,7642.1363,0.1429
1,3,6721.2197,0.1510
2,4,6243.0819,0.1256
3,5,5857.3303,0.1234
4,6,5576.0495,0.1230
5,7,5366.2767,0.1149


For storytelling, k=4 is used to create interpretable digital lifestyle archetypes.


In [11]:
# Fit KMeans and name clusters
title("Fit KMeans and name clusters")
kmeans = KMeans(n_clusters=4, random_state=42, n_init=20)
df["Cluster"] = kmeans.fit_predict(X_scaled)
profile = df.groupby("Cluster")[features].mean().round(2)
display(profile)

# Automatic cluster naming based on strongest behavioral trait.
cluster_names = {}
for cid, row in profile.iterrows():
    if row["Daily_Screen_Time_Hours"] < profile["Daily_Screen_Time_Hours"].median() and row["Number_of_Apps_Used"] < profile["Number_of_Apps_Used"].median():
        name = "Digital Minimalists"
    elif row["Social_Media_Usage_Hours"] >= max(row["Productivity_App_Usage_Hours"], row["Gaming_App_Usage_Hours"]):
        name = "Social Enthusiasts"
    elif row["Productivity_App_Usage_Hours"] >= max(row["Social_Media_Usage_Hours"], row["Gaming_App_Usage_Hours"]):
        name = "Productivity Focused"
    else:
        name = "Mobile Gamers"
    cluster_names[cid] = name
    
# avoid duplicate names
seen, final = {}, {}
for cid, name in cluster_names.items():
    seen[name] = seen.get(name, 0) + 1
    final[cid] = name if seen[name] == 1 else f"{name} {seen[name]}"
df["Cluster_Name"] = df["Cluster"].map(final)

display(df["Cluster_Name"].value_counts().to_frame("Users"))
display(df.groupby("Cluster_Name")[features].mean().round(2))



Fit KMeans and name clusters


,Age,Daily_Screen_Time_Hours,Total_App_Usage_Hours,Number_of_Apps_Used,Social_Media_Usage_Hours,Productivity_App_Usage_Hours,Gaming_App_Usage_Hours,Entertainment_Hours,App_Fragmentation_Score
Cluster,,,,,,,,,
0,39.18,8.05,5.27,18.25,2.82,2.58,4.03,6.86,0.33
1,37.03,7.88,6.97,19.21,3.74,2.04,1.58,5.32,0.40
2,38.19,7.22,8.59,6.11,2.51,2.43,2.53,5.04,1.59
3,39.84,7.43,6.13,17.69,1.13,2.77,1.65,2.79,0.40


,Users
Cluster_Name,
Digital Minimalists 2,322
Mobile Gamers,303
Social Enthusiasts,239
Digital Minimalists,136


,Age,Daily_Screen_Time_Hours,Total_App_Usage_Hours,Number_of_Apps_Used,Social_Media_Usage_Hours,Productivity_App_Usage_Hours,Gaming_App_Usage_Hours,Entertainment_Hours,App_Fragmentation_Score
Cluster_Name,,,,,,,,,
Digital Minimalists,38.19,7.22,8.59,6.11,2.51,2.43,2.53,5.04,1.59
Digital Minimalists 2,39.84,7.43,6.13,17.69,1.13,2.77,1.65,2.79,0.40
Mobile Gamers,39.18,8.05,5.27,18.25,2.82,2.58,4.03,6.86,0.33
Social Enthusiasts,37.03,7.88,6.97,19.21,3.74,2.04,1.58,5.32,0.40


In [12]:
# PCA user map
title("PCA user map")
pca = PCA(n_components=2, random_state=42)
pts = pca.fit_transform(X_scaled)
df["PCA_1"] = pts[:, 0]
df["PCA_2"] = pts[:, 1]
print(f"PCA 1 variance: {pca.explained_variance_ratio_[0]*100:.2f}%")
print(f"PCA 2 variance: {pca.explained_variance_ratio_[1]*100:.2f}%")
fig = px.scatter(df, x="PCA_1", y="PCA_2", color="Cluster_Name", opacity=0.75, hover_data=["Age", "Gender", "Location", "Daily_Screen_Time_Hours"], title="PCA user map — behavioral space")
fig.update_layout(height=650)
fig.show()



PCA user map
PCA 1 variance: 22.49%
PCA 2 variance: 19.65%


In [13]:
# Cluster profiles and advanced visual prototypes
title("Cluster profiles and advanced visual prototypes")
profile_cols = ["Social_Media_Usage_Hours", "Productivity_App_Usage_Hours", "Gaming_App_Usage_Hours", "Daily_Screen_Time_Hours", "Total_App_Usage_Hours", "Number_of_Apps_Used"]
long_profile = df.groupby("Cluster_Name")[profile_cols].mean().reset_index().melt(id_vars="Cluster_Name", var_name="Metric", value_name="Average")
fig = px.bar(long_profile, x="Metric", y="Average", color="Cluster_Name", barmode="group", title="Cluster profile comparison")
fig.update_layout(xaxis_tickangle=-30, height=560)
fig.show()

# Explore interesting relationships within clusters
fig = px.density_contour(df, x="Daily_Screen_Time_Hours", y="Total_App_Usage_Hours", color="Cluster_Name", title="Density contours — screen time vs total app usage")
fig.update_traces(contours_coloring="fill", contours_showlabels=True)
fig.show()

# Parallel coordinates for high-dimensional behavior patterns
sample = df.sample(min(500, len(df)), random_state=42).copy()
codes = {n: i for i, n in enumerate(sorted(df["Cluster_Name"].unique()))}
sample["Cluster_Code"] = sample["Cluster_Name"].map(codes)
fig = px.parallel_coordinates(sample, dimensions=["Age", "Number_of_Apps_Used", "Daily_Screen_Time_Hours", "Social_Media_Usage_Hours", "Productivity_App_Usage_Hours", "Gaming_App_Usage_Hours", "Total_App_Usage_Hours"], color="Cluster_Code", title="Parallel coordinates — high-dimensional user behavior")
fig.show()



Cluster profiles and advanced visual prototypes


In [14]:
# Export advanced analytical dataset
title("Export advanced analytical dataset")
df.to_csv(PROCESSED_DIR / "smartphone_clustered.csv", index=False)
df.groupby("Cluster_Name")[profile_cols].mean().round(3).reset_index().to_csv(PROCESSED_DIR / "cluster_profiles.csv", index=False)
size = df["Cluster_Name"].value_counts().reset_index()
size.columns = ["Cluster_Name", "Users"]
size["Percentage"] = size["Users"] / len(df)
size.to_csv(PROCESSED_DIR / "cluster_sizes.csv", index=False)
print("Saved smartphone_clustered.csv, cluster_profiles.csv, cluster_sizes.csv")



Export advanced analytical dataset
Saved smartphone_clustered.csv, cluster_profiles.csv, cluster_sizes.csv
